# MCI-GRU Experiment Workflow - Google Colab

This notebook demonstrates the complete workflow for running MCI-GRU experiments with persistent storage to Google Drive.

**Features:**
- Automatic output organization with timestamps
- Training logs saved to files
- Comprehensive backtest results (CSV, JSON, plots, time series)
- All outputs synced to Google Drive
- Easy experiment comparison

## 1. Setup: Mount Google Drive and Install Dependencies

In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Setup experiment directory in Google Drive
GDRIVE_BASE = '/content/drive/MyDrive/MCI-GRU-Experiments'
os.makedirs(GDRIVE_BASE, exist_ok=True)

print(f"✓ Google Drive mounted")
print(f"✓ Experiment directory: {GDRIVE_BASE}")
print(f"✓ All outputs will be automatically saved here")

In [ ]:
# Clone repository (if not already done)
if not os.path.exists('/content/MCI-GRU'):
    !git clone https://github.com/yourusername/MCI-GRU.git /content/MCI-GRU
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

# Change to project directory
%cd /content/MCI-GRU

# Install requirements
!pip install -q -r requirements.txt
print("✓ Dependencies installed")

## 2. Data Preparation

Upload your data files to the repository directory or download them programmatically.

In [ ]:
# Check if data files exist
import os

data_files = ['sp500_yf_download.csv', 'sp500_data.csv']
for df in data_files:
    if os.path.exists(df):
        print(f"✓ Found: {df}")
    else:
        print(f"✗ Missing: {df} - Please upload this file")

## 3. Run Training Experiment

Train models with automatic logging and output management.

In [ ]:
# Run baseline experiment
# All outputs will be saved to Google Drive with timestamp

!python run_experiment.py \
    output_dir={GDRIVE_BASE} \
    experiment_name=baseline \
    training.num_epochs=100 \
    training.num_models=10

print("\n" + "="*80)
print("✓ Training complete!")
print(f"Check outputs in: {GDRIVE_BASE}/baseline/")
print("="*80)

### Run Alternative Experiments

Test different configurations easily:

In [ ]:
# Experiment with VIX features
!python run_experiment.py \
    output_dir={GDRIVE_BASE} \
    +experiment=with_vix \
    training.num_epochs=100

In [ ]:
# Experiment with different lookback period
!python run_experiment.py \
    output_dir={GDRIVE_BASE} \
    experiment_name=lookback_20 \
    model.his_t=20 \
    training.num_epochs=100

## 4. Find Latest Experiment Run

In [ ]:
import glob
import os

def find_latest_run(experiment_name):
    """Find the most recent run for an experiment."""
    experiment_dir = f"{GDRIVE_BASE}/{experiment_name}"
    
    # Look for timestamped directories
    run_dirs = sorted(glob.glob(f"{experiment_dir}/*/"))
    
    if run_dirs:
        latest_run = run_dirs[-1]
        print(f"Latest run: {latest_run}")
        return latest_run
    elif os.path.exists(experiment_dir):
        print(f"Using experiment directory: {experiment_dir}")
        return experiment_dir
    else:
        print(f"No runs found for experiment: {experiment_name}")
        return None

# Find latest baseline run
LATEST_RUN = find_latest_run('baseline')

if LATEST_RUN:
    PREDICTIONS_PATH = f"{LATEST_RUN}averaged_predictions"
    print(f"\nPredictions path: {PREDICTIONS_PATH}")
    
    # Check what was generated
    if os.path.exists(f"{LATEST_RUN}"):
        print(f"\nGenerated files:")
        !ls -lh {LATEST_RUN}

## 5. Run Backtesting with Auto-Save

Evaluate predictions with comprehensive output saving.

In [ ]:
# Backtest WITHOUT transaction costs
print("Running backtest WITHOUT transaction costs...\n")

!python evaluate_sp500.py \
    --predictions_dir {PREDICTIONS_PATH} \
    --auto_save \
    --plot

print("\n✓ Backtest complete (no transaction costs)")
print(f"Results saved to: {LATEST_RUN}backtest/")

In [ ]:
# Backtest WITH transaction costs (separate output directory)
print("Running backtest WITH transaction costs...\n")

!python evaluate_sp500.py \
    --predictions_dir {PREDICTIONS_PATH} \
    --auto_save \
    --backtest_suffix _with_costs \
    --transaction_costs \
    --spread 10 \
    --slippage 5 \
    --plot

print("\n✓ Backtest complete (with transaction costs)")
print(f"Results saved to: {LATEST_RUN}backtest_with_costs/")

## 6. View Backtest Results

In [ ]:
# View summary text file
summary_file = f"{LATEST_RUN}backtest/summary.txt"
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        print(f.read())
else:
    print("Summary file not found")

In [ ]:
# Load and display results DataFrame
import pandas as pd

results_file = f"{LATEST_RUN}backtest/backtest_results.csv"
if os.path.exists(results_file):
    results_df = pd.read_csv(results_file)
    display(results_df.T)  # Transpose for easier reading
else:
    print("Results file not found")

In [ ]:
# Display equity curve
from IPython.display import Image, display

equity_plot = f"{LATEST_RUN}backtest/equity_curve.png"
if os.path.exists(equity_plot):
    display(Image(filename=equity_plot))
else:
    print("Equity curve plot not found")

## 7. Compare Multiple Experiments

In [ ]:
import pandas as pd
import os

def load_experiment_results(experiment_name, with_costs=False):
    """Load backtest results from an experiment."""
    latest_run = find_latest_run(experiment_name)
    if not latest_run:
        return None
    
    suffix = '_with_costs' if with_costs else ''
    results_file = f"{latest_run}backtest{suffix}/backtest_results.csv"
    
    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        df['experiment'] = experiment_name
        df['transaction_costs'] = with_costs
        return df
    return None

# Compare experiments
experiments = ['baseline', 'with_vix', 'lookback_20']
all_results = []

print("Loading experiment results...\n")
for exp in experiments:
    # Without transaction costs
    result = load_experiment_results(exp, with_costs=False)
    if result is not None:
        all_results.append(result)
        print(f"✓ {exp} (no costs)")
    
    # With transaction costs
    result = load_experiment_results(exp, with_costs=True)
    if result is not None:
        all_results.append(result)
        print(f"✓ {exp} (with costs)")

if all_results:
    comparison_df = pd.concat(all_results, ignore_index=True)
    
    # Select key metrics
    key_metrics = ['experiment', 'transaction_costs', 'ARR', 'AVoL', 'ASR', 'MDD', 'CR', 'IR', 'MSE', 'MAE']
    comparison_df = comparison_df[key_metrics]
    
    print("\n" + "="*100)
    print("EXPERIMENT COMPARISON")
    print("="*100)
    display(comparison_df)
    
    # Save comparison
    comparison_file = f"{GDRIVE_BASE}/experiment_comparison.csv"
    comparison_df.to_csv(comparison_file, index=False)
    print(f"\n✓ Comparison saved to: {comparison_file}")
else:
    print("No results found to compare")

## 8. Visualize Performance Comparison

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if all_results:
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # ARR comparison
    ax1 = axes[0, 0]
    comparison_df.plot(x='experiment', y='ARR', kind='bar', ax=ax1, legend=False)
    ax1.set_title('Annualized Return (ARR)')
    ax1.set_ylabel('ARR')
    ax1.axhline(y=0, color='r', linestyle='--', alpha=0.3)
    
    # Sharpe Ratio comparison
    ax2 = axes[0, 1]
    comparison_df.plot(x='experiment', y='ASR', kind='bar', ax=ax2, legend=False)
    ax2.set_title('Annualized Sharpe Ratio (ASR)')
    ax2.set_ylabel('ASR')
    ax2.axhline(y=2.0, color='g', linestyle='--', alpha=0.3, label='Target: 2.0')
    ax2.legend()
    
    # Maximum Drawdown comparison
    ax3 = axes[1, 0]
    comparison_df.plot(x='experiment', y='MDD', kind='bar', ax=ax3, legend=False)
    ax3.set_title('Maximum Drawdown (MDD)')
    ax3.set_ylabel('MDD')
    ax3.axhline(y=0, color='r', linestyle='--', alpha=0.3)
    
    # Information Ratio comparison
    ax4 = axes[1, 1]
    comparison_df.plot(x='experiment', y='IR', kind='bar', ax=ax4, legend=False)
    ax4.set_title('Information Ratio (IR)')
    ax4.set_ylabel('IR')
    ax4.axhline(y=0, color='r', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    
    # Save comparison plot
    comparison_plot = f"{GDRIVE_BASE}/experiment_comparison.png"
    plt.savefig(comparison_plot, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Comparison plot saved to: {comparison_plot}")

## 9. View Daily Returns Time Series

In [ ]:
# Load and plot daily returns
returns_file = f"{LATEST_RUN}backtest/daily_returns.csv"

if os.path.exists(returns_file):
    returns_df = pd.read_csv(returns_file)
    returns_df['date'] = pd.to_datetime(returns_df['date'])
    
    # Plot cumulative returns
    returns_df['cum_portfolio'] = (1 + returns_df['portfolio_return']).cumprod()
    returns_df['cum_benchmark'] = (1 + returns_df['benchmark_return']).cumprod()
    
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Cumulative returns
    ax1 = axes[0]
    ax1.plot(returns_df['date'], returns_df['cum_portfolio'], label='Portfolio', linewidth=2)
    ax1.plot(returns_df['date'], returns_df['cum_benchmark'], label='Benchmark', linewidth=2)
    ax1.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Daily returns
    ax2 = axes[1]
    ax2.plot(returns_df['date'], returns_df['portfolio_return'], alpha=0.7, label='Portfolio Daily Return')
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.3)
    ax2.set_title('Daily Returns', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Daily Return')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display statistics
    print("\nDaily Returns Statistics:")
    print("="*50)
    print(returns_df[['portfolio_return', 'benchmark_return', 'excess_return']].describe())
else:
    print("Daily returns file not found")

## 10. Access Files from Google Drive

All your outputs are now in Google Drive and can be accessed from any device!

In [ ]:
# List all experiment outputs
print("Your experiment outputs in Google Drive:")
print("="*80)
!ls -lh {GDRIVE_BASE}

print("\n" + "="*80)
print("Access these files from Google Drive at:")
print(f"https://drive.google.com/drive/folders/[Your MCI-GRU-Experiments folder]")
print("="*80)